[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_04_data_science_math/ch18_data_wrangling_surveys/03_coding_exam_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 18 Hands-On Coding Exam — Real-World Survey & Surveillance Wrangling (100 Points)

**Book Reference:** Chapter 18, Section 18.7 (Hands-On Coding Exam)  
**Suggested Time:** 40–50 minutes (Open Book, Closed Solution)

| Task | Data-Wrangling Function | Points |
| :--- | :--- | :---: |
| **Task 18.1** | `build_clean_dhs_analytic_table(mch_df)` (Normalize, neutralize `999`, group-median Hb, `anc_4plus`) | 30 |
| **Task 18.2** | `equity_gap_by_residence_wealth(clean_mch_df)` (Stratified flat table, 1-dec %, sorted) | 35 |
| **Task 18.3** | `prepare_epi_time_features(epi_df)` (Incidence/100k, leak-free 1-wk lag & 4-wk rolling mean) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Exam Auto-Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
import sharukh_tutor as tutor

tutor.activate(chapter="ch18", notebook="coding_exam")


---
## Exam Task 18.1 (30 Points): `build_clean_dhs_analytic_table(mch_df)`

Implement `build_clean_dhs_analytic_table(mch_df)` returning a **non-mutating copy** of `mch_df`:
1. Normalize `residence` casing and whitespace via `.str.strip().str.title()`.
2. Replace DHS sentinel `anc_visits == 999` with `np.nan`.
3. Cast `maternal_hb_g_dl` to numeric (`pd.to_numeric(..., errors="coerce")`) and impute missing `maternal_hb_g_dl` values with the per-`residence` median (using the **cleaned** `residence` column!).
4. Add boolean column `anc_4plus` equal to `True` when `anc_visits >= 4` and `False` otherwise (including `False` for `NaN` / former `999`).

In [ ]:
def build_clean_dhs_analytic_table(mch_df):
    """Clean DHS survey, impute Hb by residence, and flag anc_4plus."""
    out = mch_df.copy()
    out["residence"] = out["residence"].str.strip().str.title()
    out["anc_visits"] = out["anc_visits"].replace(999, np.nan)
    out["maternal_hb_g_dl"] = pd.to_numeric(
        out["maternal_hb_g_dl"], errors="coerce"
    )
    medians = out.groupby("residence")["maternal_hb_g_dl"].transform(
        "median"
    )
    out["maternal_hb_g_dl"] = out["maternal_hb_g_dl"].fillna(medians)
    out["anc_4plus"] = (
        (out["anc_visits"] >= 4).fillna(False).astype(bool)
    )
    return out


---
## Exam Task 18.2 (35 Points): `equity_gap_by_residence_wealth(clean_mch_df)`

Implement `equity_gap_by_residence_wealth(clean_mch_df)`:
1. Group by `["residence", "wealth_quintile"]` with `as_index=False`.
2. Compute columns:
   - `n_households`: household count per stratum.
   - `stunting_pct`: `child_stunted` mean $\times 100$, rounded to **1 decimal place**.
   - `anemia_pct`: `maternal_anemia` mean $\times 100$, rounded to **1 decimal place**.
   - `anc4_pct`: `anc_4plus` mean $\times 100$, rounded to **1 decimal place**.
3. Sort by `["residence", "wealth_quintile"]` ascending and return the resulting DataFrame without mutating `clean_mch_df`.

In [ ]:
def equity_gap_by_residence_wealth(clean_mch_df):
    """Stratify stunting, anemia, and ANC4+ by residence and wealth."""
    out = clean_mch_df.groupby(
        ["residence", "wealth_quintile"], as_index=False
    ).agg(
        n_households=("child_stunted", "size"),
        stunting_pct=("child_stunted", "mean"),
        anemia_pct=("maternal_anemia", "mean"),
        anc4_pct=("anc_4plus", "mean"),
    )
    for col in ("stunting_pct", "anemia_pct", "anc4_pct"):
        out[col] = (out[col] * 100).round(1)
    return out.sort_values(
        ["residence", "wealth_quintile"]
    ).reset_index(drop=True)


---
## Exam Task 18.3 (35 Points): `prepare_epi_time_features(epi_df)`

Implement `prepare_epi_time_features(epi_df)` returning a **non-mutating copy** sorted by `["district", "year", "epi_week"]` with three engineered time-series columns:
1. `incidence_per_100k`: `(dengue_cases / population) * 100_000`, rounded to **2 decimal places**.
2. `prev_week_cases`: per-district 1-week lag of `dengue_cases` via `.groupby("district")["dengue_cases"].shift(1)` (first week of every district must be `NaN` — zero cross-district leakage!).
3. `rolling_4w_cases`: per-district 4-week rolling mean of `dengue_cases` with `min_periods=1`, rounded to **2 decimal places** (zero cross-district leakage!).

In [ ]:
def prepare_epi_time_features(epi_df):
    """Engineer incidence, 1-week lag, and 4-week rolling dengue mean."""
    out = epi_df.sort_values(["district", "year", "epi_week"]).copy()
    out["incidence_per_100k"] = (
        out["dengue_cases"] / out["population"] * 100_000
    ).round(2)
    out["prev_week_cases"] = (
        out.groupby("district")["dengue_cases"].shift(1)
    )
    out["rolling_4w_cases"] = (
        out.groupby("district")["dengue_cases"]
        .transform(lambda s: s.rolling(window=4, min_periods=1).mean())
        .round(2)
    )
    return out


In [ ]:
exam_submissions = {
    "ch18_exam_01": build_clean_dhs_analytic_table,
    "ch18_exam_02": equity_gap_by_residence_wealth,
    "ch18_exam_03": prepare_epi_time_features,
}

tutor.check_exam("ch18", exam_submissions)


---
## Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 18.1 (Pipeline Order Bugs):** Two subtle bugs ruin DHS tables: (1) computing `anc_4plus = anc_visits >= 4` *before* replacing `999` with `np.nan` falsely labels unknown visits as `4+` (`999 >= 4` is `True`!); (2) imputing hemoglobin *before* normalizing `"rural"` to `"Rural"` splits rural respondents into two groups and produces wrong medians.
- **Task 18.2 (Flat Tidy Output):** Omitting `as_index=False` creates a two-level MultiIndex that breaks downstream `.sort_values(["residence", "wealth_quintile"])` or plotting calls expecting regular columns.
- **Task 18.3 (Cross-District Boundary Leakage):** Calling `out["dengue_cases"].shift(1)` or `.rolling(4).mean()` globally leaks Chattogram's final week of 2025 into Dhaka's Week 1 of 2024. Always group by `"district"` after sorting by `["district", "year", "epi_week"]`.